In [1]:
import glob

bvh_dir = "bvh_set"
bvh_files = sorted(glob.glob(f"{bvh_dir}/**/*.bvh", recursive=True))

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from data_loaders.data_loader import MotionClipDataset

In [3]:
dataset = MotionClipDataset(
    bvh_dir=bvh_dir,
)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True, num_workers=1)

Found 40 BVH files in bvh_set
Scanning files and creating clip index...


100%|██████████| 40/40 [00:00<00:00, 17766.83it/s]

Total 234094 trainable clips found.


In [1]:
from bvh_tools.bvh_controller import parse_bvh, get_preorder_joint_list
from bvh_tools.reverse import tensor_to_kinematics, populate_kinematics_dfs
from data_loaders.data_sampler import get_data
from bvh_tools.bvh_controller import VirtualRootJoint
from copy import deepcopy

bvh_dir = "bvh_set/Aeroplane/Aeroplane_BR.bvh"
root, motion = parse_bvh(bvh_dir)
joint_order = get_preorder_joint_list(root)
motion.build_quaternion_frames(joint_order)
virtual_root = motion.apply_virtual(root)
tensor = get_data(
    motion=motion,
    virtual_root=virtual_root,
    time_size=1, start_frame=0)

adjust_root, all_motion_frames = tensor_to_kinematics(tensor, "data/template.bvh")
populate_kinematics_dfs(adjust_root, all_motion_frames[0])
basis = deepcopy(virtual_root)


[-40.46701   0.      125.5105 ] [0. 0.]
(1, 142)
Parsing skeleton hierarchy (including End Sites)...
28
[0. 0. 0.]


In [2]:
from bvh_tools.bvh_controller import parse_bvh, get_preorder_joint_list
from bvh_tools.reverse import tensor_to_kinematics, populate_kinematics_dfs
from data_loaders.data_sampler import get_data
from bvh_tools.bvh_controller import VirtualRootJoint

bvh_dir = "bvh_set/Aeroplane/Aeroplane_BR.bvh"
root, motion = parse_bvh(bvh_dir)
joint_order = get_preorder_joint_list(root)
motion.build_quaternion_frames(joint_order)
virtual_root = motion.apply_virtual(root)
tensor = get_data(
    motion=motion,
    virtual_root=virtual_root,
    time_size=300, start_frame=0)

adjust_root, all_motion_frames = tensor_to_kinematics(tensor, "data/template.bvh")
populate_kinematics_dfs(adjust_root, all_motion_frames[299])


[-40.46701   0.      125.5105 ] [0. 0.]
[-40.49657   0.      125.37138] [-0.02956009 -0.13911438]
[-40.526127   0.       125.23227 ] [-0.02955627 -0.13911438]
[-40.560658   0.       125.08806 ] [-0.03453064 -0.14421082]
[-40.602097   0.       124.93885 ] [-0.04143906 -0.14920807]
[-40.648514   0.       124.784546] [-0.04641724 -0.1543045 ]
[-40.699085   0.       124.61973 ] [-0.05057144 -0.16481781]
[-40.753353   0.       124.44297 ] [-0.05426788 -0.17675781]
[-40.811783   0.       124.2557  ] [-0.05842972 -0.18727112]
[-40.874958   0.       124.05677 ] [-0.0631752  -0.19892883]
[-40.942654   0.       123.84624 ] [-0.06769562 -0.21053314]
[-41.0151    0.      123.62405] [-0.07244492 -0.22219086]
[-41.09355   0.      123.38892] [-0.07845306 -0.23512268]
[-41.17933   0.      123.14203] [-0.08577728 -0.24689484]
[-41.27112   0.      122.8822 ] [-0.09178925 -0.25982666]
[-41.366238   0.       122.60469 ] [-0.09511948 -0.2775116 ]
[-41.464985   0.       122.3091  ] [-0.09874725 -0.29559326]

In [3]:
basis.kinematics

mat4x4(( 0.111561, -2.68361e-09, -0.993757, 0 ), ( 8.42531e-09, 1, -1.75462e-09, 0 ), ( 0.993757, -8.17697e-09, 0.111561, 0 ), ( -40.467, 0, 125.51, 1 ))

In [7]:
adjust_root.kinematics,

(mat4x4(( -0.317949, 0, -0.948108, 0 ), ( 0, 1, 0, 0 ), ( 0.948108, 0, -0.317949, 0 ), ( 232.861, 0, -25.6715, 1 )),)

In [4]:
print(adjust_root.kinematics * basis.kinematics, adjust_root.name)
print("-------------")
print(virtual_root.kinematics, virtual_root.name)

[     -0.97766 ][ -4.34239e-09 ][    -0.210192 ][       150.84 ]
[ -2.68361e-09 ][            1 ][ -8.17697e-09 ][            0 ]
[     0.210192 ][ -7.43022e-09 ][     -0.97766 ][     -258.959 ]
[            0 ][            0 ][            0 ][            1 ] VirtualRoot
-------------
[     -0.97766 ][ -1.86487e-09 ][    -0.210192 ][     -21.4914 ]
[ -1.86487e-09 ][            1 ][ -1.98205e-10 ][            0 ]
[     0.210192 ][  1.98205e-10 ][     -0.97766 ][      -131.91 ]
[            0 ][            0 ][            0 ][            1 ] VirtualRoot


In [38]:
populate_kinematics_dfs(adjust_root, all_motion_frames[0])

In [33]:
from data_loaders.data_sampler import get_statistics

In [13]:
import numpy as np
from data_loaders.data_loader import get_data
from bvh_tools.bvh_controller import parse_bvh, get_preorder_joint_list

# 이 함수는 각 워커(프로세스)에서 실행될 작업 단위입니다.
def process_file_for_stats(file_path, clip_length, feature_dim):
    # (주의) 이 함수는 독립적인 프로세스에서 실행되므로, 필요한 모든 모듈을 내부에서 import하거나
    # 미리 import 되어 있어야 합니다.

    print(f"Processing file: {file_path}")
    
    local_count = 0
    local_sum = np.zeros(feature_dim)
    local_sum_sq = np.zeros(feature_dim) # 제곱의 합

    try:
        root, motion = parse_bvh(file_path)
        joint_order = get_preorder_joint_list(root)
        motion.build_quaternion_frames(joint_order)
        virtual_root = motion.apply_virtual(root)
        
        if motion.frames >= clip_length:
            for start_frame in range(motion.frames - clip_length + 1):
                clip_data = get_data(motion, virtual_root, clip_length, start_frame)
                
                # 자신만의 통계치를 누적
                local_count += clip_data.shape[0] 
                local_sum += np.sum(clip_data, axis=0)
                local_sum_sq += np.sum(clip_data**2, axis=0)

    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        
    print(f"Finished processing file: {file_path}")
    return local_count, local_sum, local_sum_sq

In [14]:
from joblib import Parallel, delayed
from tqdm import tqdm

def get_statistics_parallel(bvh_files, clip_length=180, feature_dim=172):
    print("병렬 처리를 시작합니다...")
    
    with Parallel(n_jobs=-1) as parallel:
        
        # 1. 실행할 작업 목록을 미리 준비합니다.
        tasks = (delayed(process_file_for_stats)(fp, clip_length, feature_dim) for fp in bvh_files)
        
        # 2. parallel()이 반환하는 결과 이터레이터(iterator)를 tqdm으로 감쌉니다.
        results = tqdm(parallel(tasks), total=len(bvh_files))
        
        # 3. 결과를 리스트로 변환합니다. 이 과정에서 프로그레스 바가 표시됩니다.
        results_list = list(results)
    
    # 각 워커가 반환한 결과들을 취합
    total_count = 0
    total_sum = np.zeros(feature_dim)
    total_sum_sq = np.zeros(feature_dim)
    
    for local_count, local_sum, local_sum_sq in results:
        total_count += local_count
        total_sum += local_sum
        total_sum_sq += local_sum_sq
        
    if total_count == 0:
        print("처리된 데이터가 없습니다.")
        return

    # 최종 평균과 표준편차 계산
    mean = total_sum / total_count
    # 분산 = (제곱의 평균) - (평균의 제곱)
    variance = (total_sum_sq / total_count) - (mean ** 2)
    std = np.sqrt(variance)

    print("\n계산 완료!")
    print(f"Mean shape: {mean.shape}")
    print(f"Std shape: {std.shape}")

    # 파일 저장
    np.save('data/mean.npy', mean)
    np.save('data/std.npy', std)

# --- 실행 ---
get_statistics_parallel(bvh_files)

병렬 처리를 시작합니다...


Processing file: bvh_set/Aeroplane/Aeroplane_BR.bvh
Processing file: bvh_set/Aeroplane/Aeroplane_BW.bvh
Processing file: bvh_set/Aeroplane/Aeroplane_FR.bvh
Processing file: bvh_set/Aeroplane/Aeroplane_FW.bvh
Processing file: bvh_set/Aeroplane/Aeroplane_ID.bvh
Processing file: bvh_set/Aeroplane/Aeroplane_SR.bvh
Processing file: bvh_set/Aeroplane/Aeroplane_SW.bvh
Processing file: bvh_set/Aeroplane/Aeroplane_TR1.bvh
Processing file: bvh_set/Cat/Cat_BR.bvh
Processing file: bvh_set/Cat/Cat_BW.bvh
Processing file: bvh_set/Cat/Cat_FR.bvh
Processing file: bvh_set/Cat/Cat_FW.bvh
Processing file: bvh_set/Cat/Cat_ID.bvh
Processing file: bvh_set/Cat/Cat_SR.bvhProcessing file: bvh_set/Cat/Cat_SW.bvh

Processing file: bvh_set/Cat/Cat_TR1.bvh
Processing file: bvh_set/Dinosaur/Dinosaur_BR.bvh
Processing file: bvh_set/Dinosaur/Dinosaur_BW.bvh
Processing file: bvh_set/Dinosaur/Dinosaur_FR.bvh
Processing file: bvh_set/Dinosaur/Dinosaur_ID.bvh
Processing file: bvh_set/Dinosaur/Dinosaur_SR.bvh
Processing f

100%|██████████| 40/40 [00:00<00:00, 234646.38it/s]


계산 완료!
Mean shape: (172,)
Std shape: (172,)
